# QwerySmith v3.0 - Eval Matrix (serving + eval + report)

Serves the four-system matrix and runs `eval` + `report` on identical questions and identical frozen packs. **Runtime → L4 GPU.**

Rows 1-2 (8B base + LoRA adapter) and row 3 (30B-A3B AWQ) run as vLLM servers; the frontier row is an API call. The harness treats them identically - same prompts, same frozen packs, same scorer.

| Cell | What it does |
|---|---|
| 1 | Setup + artifact restore from Drive |
| 2 | Start 8B server (base + current seed adapter) - background vLLM |
| 3 | Run eval pass for the current seed + baseline/onprem/reference |
| 4 | Loop seeds 2, 3 (restart 8B server per seed) |
| 5 | Report: aggregate 3 seeds, gate verdict, failure folders |
| 6 | Zip results to Drive |

In [ ]:
# restore adapters + prepared artifacts from the training run
zips = sorted(glob.glob(f'{DATA}/run_*_train_*.zip'))
assert zips, 'no training run zip in Drive - run t4_train.ipynb first'
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(REPO)
print('restored:', zips[-1])

In [ ]:
# ===== Cell 2: start 8B server (base + current seed adapter) =====
# vLLM in the background; one adapter served at a time under the stable
# name 'ft' - systems.yaml model_id never changes across seed passes
SEED = 1   # set to 2 or 3 before re-running this cell in the seed loop

def stop_servers():
    """Terminate the running vLLM process and WAIT for VRAM to come back -
    pkill+sleep alone races the next server load (OOM class fixed in training)."""
    import time
    srv = globals().get('srv')
    if srv is not None and srv.poll() is None:
        srv.terminate()
        try:
            srv.wait(timeout=120)
        except subprocess.TimeoutExpired:
            srv.kill(); srv.wait()
    time.sleep(10)
    total = torch.cuda.get_device_properties(0).total_memory
    for _ in range(36):   # up to 3 min: CUDA memory returns slowly
        free, _ = torch.cuda.mem_get_info()
        if free > 0.80 * total:
            break
        time.sleep(5)
    free, _ = torch.cuda.mem_get_info()
    print(f'VRAM free: {free / 2**20:.0f} / {total / 2**20:.0f} MB')

stop_servers()
log = open(f'/content/vllm_8b_seed{SEED}.log', 'w')
srv = subprocess.Popen(
    ['vllm', 'serve', 'Qwen/Qwen3-8B', '--port', '8000', '--enable-lora',
     f'--lora-modules', f'ft={run_dir}/adapters/adapter_seed{SEED}',
     '--max-lora-rank', '16', '--gpu-memory-utilization', '0.55',
     '--max-model-len', '8192'],
    stdout=log, stderr=subprocess.STDOUT)

import time, urllib.request, json
for _ in range(120):   # up to 10 min for model download + warmup
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print(f'8B server up (adapter_seed{SEED})'); break
    except Exception:
        continue
else:
    print(open(f'/content/vllm_8b_seed{SEED}.log').read()[-3000:]); raise SystemExit('8B server never became healthy')

In [ ]:
# ===== Cell 3: one eval pass - baseline + candidate(seed) + onprem + frontier =====
# systems.yaml: this pass uses the local 8B server. On a single L4, the 30B
# row runs as a SECOND pass after this one (see Cell 4 note) - keep
# gpu-memory-utilization low here so the 8B + 30B never co-reside.
import yaml
sysyaml_path = f'{REPO}/datasets/olist/systems.yaml'
sysyaml = yaml.safe_load(open(sysyaml_path))
for s in sysyaml['systems']:
    if s['name'] == 'qwen3-8b-base':
        s.update(base_url='http://localhost:8000/v1', model_id='Qwen/Qwen3-8B', role='baseline')
    if s['name'] == 'qwen3-8b-ft':
        s.update(base_url='http://localhost:8000/v1', model_id='ft', role='candidate_seed')
    if s['name'] == 'qwen3-30b-a3b':
        s.update(role='onprem')   # second pass flips base_url; see Cell 4

# frontier placeholder in systems.yaml (api.example.com) must be patched from
# Colab secrets BEFORE eval, else urlopen dies mid-pass. Missing secrets =>
# drop the reference row from this pass instead of crashing it.
frontier_in_pass = False
try:
    _fb = userdata.get('FRONTIER_BASE_URL')
    _fm = userdata.get('FRONTIER_MODEL_ID')
    if not _fb or not _fm:
        raise ValueError('empty')
    for s in sysyaml['systems']:
        if s['name'] == 'frontier':
            s.update(base_url=_fb, model_id=_fm, role='reference')
    frontier_in_pass = True
    print(f'frontier patched: {_fm} via {_fb}')
except Exception:
    print('FRONTIER_BASE_URL/FRONTIER_MODEL_ID secrets missing - skipping reference row')
open(sysyaml_path, 'w').write(yaml.safe_dump(sysyaml, sort_keys=False))

# this pass: baseline + candidate(seed) (+ reference if patched) - 30B runs in Cell 4
eval_roles = 'baseline,candidate_seed' + (',reference' if frontier_in_pass else '')
r = subprocess.run(['python', '-m', 'qwery_smith', 'eval', 'olist',
                    '--roles', eval_roles,
                    '--tag', f'seed{SEED}'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:]); raise SystemExit('eval pass FAILED')

In [ ]:
# ===== Cell 4: seed loop (2,3) + the 30B pass =====
# For seeds 2 and 3: set SEED in Cell 2, re-run Cells 2+3.
#
# Then the 30B pass (onprem row): stop the 8B server, serve 30B AWQ on
# the freed GPU, and run the onprem role once:
stop_servers()
log30 = open('/content/vllm_30b.log', 'w')
srv = subprocess.Popen(
    ['vllm', 'serve', 'cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',
     '--port', '8001', '--max-model-len', '8192',
     '--gpu-memory-utilization', '0.92'],
    stdout=log30, stderr=subprocess.STDOUT)
import time, urllib.request
for _ in range(120):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8001/health', timeout=2)
        print('30B server up'); break
    except Exception:
        continue
else:
    print(open('/content/vllm_30b.log').read()[-3000:]); raise SystemExit('30B server never became healthy')

import yaml
sysyaml = yaml.safe_load(open(sysyaml_path))
for s in sysyaml['systems']:
    if s['name'] == 'qwen3-30b-a3b':
        s.update(base_url='http://localhost:8001/v1',
                 model_id='cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit')
open(sysyaml_path, 'w').write(yaml.safe_dump(sysyaml, sort_keys=False))

r = subprocess.run(['python', '-m', 'qwery_smith', 'eval', 'olist',
                    '--roles', 'onprem', '--tag', 'onprem'],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:]); raise SystemExit('onprem eval FAILED')

In [ ]:
# ===== Cell 5: report - aggregate 3 seeds + gate + failure folders =====
# merges every run dir (seed1/seed2/seed3/onprem passes) under runs/olist
r = subprocess.run(['python', '-m', 'qwery_smith', 'report', 'olist',
                    '--run', f'{REPO}/runs/olist'],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-2000:]); raise SystemExit('report FAILED')

In [ ]:
# ===== Cell 6: publish the FINAL release to Hugging Face =====
# After the gate verdict exists: per-seed repos get the measured results table
# embedded, and the median-EX seed is aliased as the canonical
# Cyrax321/QwerySmith-2.0 repo. Gate FAIL is stated plainly on the card.
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# median-EX seed from the report: pick by hand from the table, pass it in.
MEDIAN_SEED = 2
GATE_PASS = False   # <- set from the Cell 5 gate verdict, honestly

# typer bools are flag pairs, never '--gate-pass False' (that exits rc=2)
gate_args = (['--gate-pass'] if GATE_PASS else ['--no-gate-pass'])
r = subprocess.run(['python', '-m', 'qwery_smith', 'publish', 'olist',
                    '--hf-user', 'Cyrax321',
                    '--canonical-seed', str(MEDIAN_SEED),
                    *gate_args],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('publish FAILED')

In [ ]:
# ===== Cell 7: zip results to Drive =====
zip_target = f'{DATA}/eval_results.zip'
subprocess.run(['zip', '-qr', zip_target, 'runs/olist',
                'datasets/olist/systems.yaml'], cwd=REPO, check=True)
print('zipped ->', zip_target)
print('deliverables: report.md (table + gate), failures/ (one file per wrong answer), raw outputs')